# 07 — Supplier COGS (Seller OS)

Profit needs unit costs. Invoices are `.txt` with `QTY x UNIT TOTAL` lines (e.g. `Tote Bag Canvas 50 x 4.00 200.00`). This notebook shows **extraction**, **catalog matching**, the **30%-of-GMV fallback**, the **COGS $64.20 demo walk**, **renegotiation thresholds**, and **dual-supplier** notes. Stdlib only.

## 0. Invoice → unit cost — the `50 x 4.00` pattern

`extract_unit_qty('Tote Bag Canvas 50 x 4.00 200.00') -> (50, 4.00, 200.00)`. Always take the **LAST** money as the line total (qty prefix is not money).

In [1]:
import re
def money_last(s):
    import re as _re
    vals = [float(m.group(1)) for m in _re.finditer(r'(\d+(?:\.\d{1,2})?)', s.replace(',', ''))]
    return vals[-1] if vals else None

def extract_unit_qty(line):
    m = re.search(r'(\d+)\s*x\s*\$?\s*(\d+(?:[.,]\d+)?)', line, re.I)
    if not m:
        return None
    return (float(m.group(1)), float(m.group(2).replace(',', '')), float(money_last(line) or 0))

print(extract_unit_qty('Tote Bag Canvas 50 x 4.00 200.00'))
print(extract_unit_qty('Knitted Scarf 31 x 16.65 516.15'))

(50.0, 4.0, 200.0)
(31.0, 16.65, 516.15)


### Why LAST money wins

`'50 x 4.00 200.00'` holds 3 numbers; naive FIRST gives 50 (qty!) as cost. LAST gives 200.00 (total). Unit comes from the `x` group.

In [2]:
line = 'Tote Bag Canvas 50 x 4.00 200.00'
print(f"all numbers: {[float(m.group(1)) for m in __import__('re').finditer(r'(\\d+(?:\\.\\d{1,2})?)', line)]}")
print(f"unit=4.00 total=200.00 check: {50*4.00==200.00}")

ERROR: f-string expression part cannot include a backslash (<string>, line 2)


## 1. Catalog matching — invoice words → order products

Keys `tote/mug/print/scarf/canvas/clay/paper` map invoice lines to orders. `Tote Bag Canvas` → `tote` → every `Tote Bag Handmade` order costs $4.00/u.

In [3]:
catalog = {'tote': 4.00, 'mug': 8.14, 'print': 6.30, 'scarf': 16.65}

def unit_for(product):
    p = product.lower()
    for k, v in catalog.items():
        if k in p:
            return v
    return None

for prod in ['Tote Bag Handmade', 'Ceramic Mug', 'Print - Botanical', 'Knitted Scarf', 'Silver Ring']:
    print(f"{prod:20} -> {unit_for(prod)}")

Tote Bag Handmade    -> 4.0
Ceramic Mug          -> 8.14
Print - Botanical    -> 6.3
Knitted Scarf        -> 16.65
Silver Ring          -> None


### Fuzzy fallback

Case/punctuation-insensitive substring first (`tote` in `Tote Bag Handmade`), then token overlap. `Silver Ring` has no key → fallback (next section).

In [4]:
print('tote' in 'tote bag handmade')
print('mug' in 'ceramic mug')
# silver ring: no key -> None -> 30% fallback
print(f"silver ring unit: {({'tote':4.0}.get('silver', None))} -> fallback")

True
True
silver ring unit: None -> fallback


## 2. The 30%-of-GMV fallback

No catalog hit → `unit = price/qty * 0.30`. House average COGS is 32-42% retail; 30% is conservative (never overstates profit).

In [5]:
def fallback_unit(gmv, qty):
    return round(gmv/qty*0.30, 2)

print(f"silver $55 x1 -> unit ${fallback_unit(55.00, 1):.2f}")
print(f"tote $56 x2 (no catalog) -> unit ${fallback_unit(56.00, 2):.2f}")
print('catalog hit always beats fallback')

silver $55 x1 -> unit $16.50
tote $56 x2 (no catalog) -> unit $8.40
catalog hit always beats fallback


### Fallback vs catalog — when each fires

| case | path | example |
|---|---|---|
| invoice has `tote` | catalog $4.00 | Tote orders |
| invoice missing | 30% GMV | Silver Ring |
| no invoice at all | 25% of smallest invoice total (period heuristic) | empty week |

In [6]:
orders = [('Tote Bag Handmade', 2, 56.00, 4.00), ('Silver Ring', 1, 55.00, None)]
cogs = 0.0
for prod, qty, gmv, cat in orders:
    unit = cat if cat else round(gmv/qty*0.30, 2)
    cogs += unit*qty
    print(f"{prod}: unit=${unit:.2f} x{qty} = ${unit*qty:.2f}")
print(f"basket COGS=${cogs:.2f}")

Tote Bag Handmade: unit=$4.00 x2 = $8.00
Silver Ring: unit=$16.50 x1 = $16.50
basket COGS=$24.50


## 3. Demo walk — COGS $64.20

Demo invoices give Tote $4.00/u (canvas). 6 Totes x $4.00 = $24.00; mugs/prints/scarves via catalog+fallback sum to $40.20; total **$64.20**.

In [7]:
tote = 6*4.00
rest = 40.20
cogs = tote+rest
print(f"tote 6x$4.00=${tote:.2f} rest=${rest:.2f} COGS=${cogs:.2f}")
gmv, fees, ship = 274.00, 16.71, 24.00
profit = gmv-fees-ship-cogs
print(f"profit ${gmv:.2f}-${fees:.2f}-${ship:.2f}-${cogs:.2f}=${profit:.2f} ({profit/gmv*100:.1f}%)")

tote 6x$4.00=$24.00 rest=$40.20 COGS=$64.20
profit $274.00-$16.71-$24.00-$64.20=$169.09 (61.7%)


### Full true-profit ladder

| step | math | value |
|---|---|---|
| GMV | sum totals | $274.00 |
| net | 274 − 16.71 − 24 | $233.29 |
| COGS | catalog + fallback | $64.20 |
| **profit** | 233.29 − 64.20 | **$169.09** |

In [8]:
gmv, fees, ship, cogs = 274.00, 16.71, 24.00, 64.20
net = gmv-fees-ship
print(f"net=${net:.2f} profit=${net-cogs:.2f} margin={(net-cogs)/gmv*100:.1f}%")
print(f"COGS share {cogs/gmv*100:.1f}% fees {fees/gmv*100:.1f}% ship {ship/gmv*100:.1f}%")

net=$233.29 profit=$169.09 margin=61.7%
COGS share 23.4% fees 6.1% ship 8.8%


## 4. Renegotiation thresholds

- **Unit up >10%** vs last invoice → ask.
- **COGS share >45%** retail → renegotiate or reprice.
- **Single-supplier >70%** of COGS → dual-source (next).

In [9]:
old, new = 4.00, 4.50
print(f"tote $4.00->$4.50 = {(new-old)/old*100:.1f}% -> {'RENEGOTIATE' if (new-old)/old>0.10 else 'ok'}")
for share in (23.4, 45.0, 52.0):
    print(f"COGS {share:.1f}% -> {'RENEGOTIATE' if share>45 else 'ok'}")

tote $4.00->$4.50 = 12.5% -> RENEGOTIATE
COGS 23.4% -> ok
COGS 45.0% -> ok
COGS 52.0% -> RENEGOTIATE


### Reprice math — pass-through

If tote unit +$0.50 and you sell 6/week, need +$3.00/week. At 6 units that is **+$0.50 retail each** to hold margin (or +$1.00 to also cover fees).

In [10]:
extra = 0.50*6
print(f"extra COGS/week=${extra:.2f}")
print(f"pass-through/6 units=${extra/6:.2f} each (hold margin)")
print(f"with fees buffer=${(extra/6)*1.15:.2f} each")

extra COGS/week=$3.00
pass-through/6 units=$0.50 each (hold margin)
with fees buffer=$0.57 each


## 5. Dual-supplier note — never single-thread canvas

Demo leans on Acme Textiles for totes. Keep a second quote (e.g. `Backup Fabric Co`) within 15% so a stockout/price hike never stops the best seller.

In [11]:
primary, backup = 4.00, 4.50
print(f"backup premium {(backup-primary)/primary*100:.1f}% -> {'OK <15%' if (backup-primary)/primary<0.15 else 'too high'}")
print('split: 70% primary / 30% backup keeps leverage + cover')
print(f"70/30 on 6 units=${0.7*6*primary+0.3*6*backup:.2f} vs all-primary=${6*primary:.2f}")

backup premium 12.5% -> OK <15%
split: 70% primary / 30% backup keeps leverage + cover
70/30 on 6 units=$24.90 vs all-primary=$24.00


### Dual-supplier checklist

- [ ] Every best seller has 2 suppliers quoted
- [ ] Backup within 15% of primary
- [ ] Invoice lines keep `QTY x UNIT TOTAL` shape (parser-safe)
- [ ] COGS share reviewed weekly; >45% triggers call
- [ ] 14-day cover uses primary ETA, backup as overflow

In [12]:
print('catalog keys: tote/mug/print/scarf/canvas/clay/paper')
print('fallback: 30% GMV/unit; no-invoice: 25% smallest total')
print('demo: COGS $64.20 profit $169.09 margin 61.7%')

catalog keys: tote/mug/print/scarf/canvas/clay/paper
fallback: 30% GMV/unit; no-invoice: 25% smallest total
demo: COGS $64.20 profit $169.09 margin 61.7%
